In [1]:
## importing necessary modules

import csv
import random
from collections import Counter
import json
import os
import pandas as pd
from openai import OpenAI

In [2]:
## generating the data

exec(open("generate v7.py").read())

Wrote 900 rows to synthetic claim business rules v7.csv
Wrote 8000 rows to synthetic communications v7.csv

Sender type mix: Counter({'insured': 4966, 'shop': 1436, 'agent': 712, 'tpc': 552, 'attorney': 258, 'lienholder': 76})
Channel mix: Counter({'email': 2707, 'text': 2625, 'call': 1601, 'app': 1067})
Calls (priority boost): 1601

Insured messages: 4966
Urgent callback requests: 1458 (29.4% of insured messages)

Fallback-to-task messages: 241
Referral-needed messages: 501
Referral category mix: Counter({'referral_billing_premium': 138, 'referral_roadside_assistance': 127, 'referral_medical_bills': 120, 'referral_glass_claims': 116})

Inspection method mix: Counter({'network_shop': 493, 'virtual_photo': 266, 'ccc1': 77, 'field': 46, 'bill_review': 18})
Supplement auto-send (virtual + non-network indicated): 65

UMPD claims: 132
  review needed: 77
  filed anyway (caused property damage, no coverage): 8
  not filed (no coverage, no property damage caused): 16

Total loss claims with a

In [3]:
## configuring the LLM, generating a sample

CSV_PATH = "synthetic communications v7.csv"
SAMPLE_SIZE = 100         
MODEL = "gpt-4.1-mini" 

client = OpenAI(
    base_url=os.environ.get("AZURE_OPENAI_BASE_URL"),
    api_key=os.environ.get("AZURE_OPENAI_BASE_KEY")
    
)  

## parsing prompt 

system_prompt = """You are a claims-intake assistant for an auto insurance company. \
You will be given one inbound message from a claimant, repair shop, attorney, \
agent, lienholder, or third-party claimant (TPC). Your job is to classify it \
so it can be routed correctly.

Return ONLY a JSON object with these fields, nothing else — no preamble, no \
markdown formatting, no code fences:

{
  "category": "<short snake_case label for what this message is about>",
  "urgency": "<low, medium, or high>",
  "auto_respond_eligible": <true or false — true only for simple, templated \
questions like supplement turnaround times, NOT for anything involving \
frustration, legal representation, injury, or coverage disputes>,
  "requires_fallback_task": <true or false — true if this is a one-off or \
unusual request that doesn't fit a standard category and needs a human \
adjuster to look at it>,
  "reasoning": "<one short sentence explaining the classification>"
}

Important rules:
- If the sender is a third-party claimant (TPC), never suggest sharing \
coverage or deductible details — TPC messages should only ever be told that \
liability needs to be determined and to send supporting evidence.
- TPC messages reporting an accident or requesting liability determination \
are MEDIUM urgency and auto_respond_eligible = true (reply asking them to \
send supporting evidence) - BUT ONLY IF none of the HIGH urgency \
conditions below apply. If a TPC message also contains anger, an \
attorney, injury, or an escalation threat, it is HIGH, not medium. \
the HIGH conditions always take precedence over this TPC default
- Mark Urgency as HIGH if ANY of the following apply, independently of \
each other:
1. Explicit anger (profanity, all-caps, words like "unacceptable" or \
"furious")
2. An attorney is mentioned as representing the claimant for injuries
3. A direct threat to escalate (e.g. "I think I need a lawyer", \
"I need to speak to a manager/supervisor" "I'm reporting this to the state")\
4. Injury is mentioned
"""


def classify_message(sender_type: str, message_body: str) -> dict:
    """Send one message to Claude and parse its JSON response."""
    user_prompt = f"Sender type: {sender_type}\nMessage: {message_body}"

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": system_prompt
            },
            {
             "role": "user",
             "content": user_prompt
            }
        ],
        temperature=0,
        max_tokens=300
    )
        

    raw_text = response.choices[0].message.content.strip()

    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        if raw_text.startswith("json"):
            raw_text = raw_text[4:]
        raw_text = raw_text.strip()

    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        return {"category": "PARSE_ERROR", "urgency": "unknown",
                 "auto_respond_eligible": False, "requires_fallback_task": True,
                 "reasoning": f"Could not parse model output: {raw_text[:200]}"}


def main():
    df = pd.read_csv(CSV_PATH, encoding="utf-8", encoding_errors="replace")
    sample = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

    results = []
    for i, row in sample.iterrows():
        print(f"Processing {i + 1}/{len(sample)}...")
        prediction = classify_message(row["sender_type"], row["message_body"])
        results.append(prediction)

    sample["predicted_category"] = [r.get("category", "unknown") for r in results]
    sample["predicted_urgency"] = [r.get("urgency", "unknown") for r in results]
    sample["predicted_auto_respond"] = [r.get("auto_respond_eligible", False) for r in results]
    sample["predicted_fallback_task"] = [r.get("requires_fallback_task", False) for r in results]
    sample["model_reasoning"] = [r.get("reasoning", "") for r in results]

## scoring
    
    urgency_correct = (sample["predicted_urgency"] == sample["true_urgency"]).sum()
    fallback_correct = (sample["predicted_fallback_task"] == sample["requires_fallback_task"]).sum()

    print("\n--- Results ---")
    print(sample[["sender_type", "message_body", "true_category", "predicted_category",
                  "true_urgency", "predicted_urgency", "model_reasoning"]].to_string())

    print(f"\nUrgency match: {urgency_correct}/{len(sample)} "
          f"({urgency_correct / len(sample) * 100:.0f}%)")
    print(f"Fallback-task match: {fallback_correct}/{len(sample)} "
          f"({fallback_correct / len(sample) * 100:.0f}%)")

    sample.to_csv("step1_parsing_results.csv", index=False)
    print("\nSaved detailed results to step1_parsing_results.csv")

    return sample

if __name__ == "__main__":
   sample = main()


Processing 1/100...
Processing 2/100...
Processing 3/100...
Processing 4/100...
Processing 5/100...
Processing 6/100...
Processing 7/100...
Processing 8/100...
Processing 9/100...
Processing 10/100...
Processing 11/100...
Processing 12/100...
Processing 13/100...
Processing 14/100...
Processing 15/100...
Processing 16/100...
Processing 17/100...
Processing 18/100...
Processing 19/100...
Processing 20/100...
Processing 21/100...
Processing 22/100...
Processing 23/100...
Processing 24/100...
Processing 25/100...
Processing 26/100...
Processing 27/100...
Processing 28/100...
Processing 29/100...
Processing 30/100...
Processing 31/100...
Processing 32/100...
Processing 33/100...
Processing 34/100...
Processing 35/100...
Processing 36/100...
Processing 37/100...
Processing 38/100...
Processing 39/100...
Processing 40/100...
Processing 41/100...
Processing 42/100...
Processing 43/100...
Processing 44/100...
Processing 45/100...
Processing 46/100...
Processing 47/100...
Processing 48/100...
P

In [4]:
## seeing what the LLM made errors on

mismatches = sample[sample["predicted_urgency"] != sample["true_urgency"]]
print(mismatches[ "predicted_urgency"].value_counts())
print(mismatches[ "true_urgency"].value_counts())
missed_highs = mismatches[mismatches["true_urgency"] == "high"]
print(missed_highs[["message_body", "predicted_urgency", "model_reasoning"]].to_string())

predicted_urgency
low       17
medium    13
Name: count, dtype: int64
true_urgency
medium    17
high      12
low        1
Name: count, dtype: int64
                                                                                                                                                                message_body predicted_urgency                                                                                               model_reasoning
0                                                                                                  Hi, #CLM100461, this is Chen Ramirez, I've left messages with no response            medium             Insured is requesting claim status with no response, a common inquiry suitable for auto-response.
5              I was the other driver in the accident with your insured on #CLM100454. Wanted to ask about liability determination and how to get my vehicle damage covered.            medium  Third-party claimant asking about liability and coverage w

In [5]:
## looking into a small sample of claims

exec(open("step3 claim briefings.py").read())

Claims with 3+ messages available: 890

[Thread path] Summarizing 1/5: CLM100580...
[Thread path] Summarizing 2/5: CLM100871...
[Thread path] Summarizing 3/5: CLM100666...
[Thread path] Summarizing 4/5: CLM100047...
[Thread path] Summarizing 5/5: CLM100362...
[FNOL-only path] Summarizing 1/5: CLM100552...
[FNOL-only path] Summarizing 2/5: CLM100157...
[FNOL-only path] Summarizing 3/5: CLM100625...
[FNOL-only path] Summarizing 4/5: CLM100169...
[FNOL-only path] Summarizing 5/5: CLM100664...


CLAIM CLM100580  [message_thread, 7 messages]
Review priority: MEDIUM   |   Contact method: EMAIL_SUFFICIENT
Summary: The claim #CLM100580 is an uninsured motorist claim with a $500 deductible and coverage limit of $25,000. The vehicle is not drivable but is now ready for pickup at multiple shops, with supplements awaiting final sign-off. Rental coverage is confirmed and ongoing, with rental ending at repair completion per standard terms. Inspection is via bill review, and there are no potential is

In [6]:
## evaluating urgency PER CLAIM

exec(open("merged task queue.py").read())

Base task queue (free, full dataset): 6237 tasks across 900 claims

Running LLM classification on a sample of 75 claims (of 900 total in queue)...

  Classifying 1/75: CLM100389...
  Classifying 2/75: CLM100311...
  Classifying 3/75: CLM100655...
  Classifying 4/75: CLM100861...
  Classifying 5/75: CLM100478...
  Classifying 6/75: CLM100830...
  Classifying 7/75: CLM100816...
  Classifying 8/75: CLM100764...
  Classifying 9/75: CLM100348...
  Classifying 10/75: CLM100698...
  Classifying 11/75: CLM100658...
  Classifying 12/75: CLM100505...
  Classifying 13/75: CLM100506...
  Classifying 14/75: CLM100443...
  Classifying 15/75: CLM100280...
  Classifying 16/75: CLM100372...
  Classifying 17/75: CLM100378...
  Classifying 18/75: CLM100341...
  Classifying 19/75: CLM100392...
  Classifying 20/75: CLM100243...
  Classifying 21/75: CLM100387...
  Classifying 22/75: CLM100676...
  Classifying 23/75: CLM100846...
  Classifying 24/75: CLM100789...
  Classifying 25/75: CLM100221...
  Classifyi

In [7]:
## evaluating urgency PER MESSAGE/TASK

exec(open("task queue v8.py").read())

Base task queue (free, full dataset): 6237 tasks qualify for review

Running LLM classification on a sample of 75 tasks (of 6237 total qualifying)...

  Classifying 1/75: MSG00753 (CLM100455)...
  Classifying 2/75: MSG02329 (CLM100393)...
  Classifying 3/75: MSG07647 (CLM100053)...
  Classifying 4/75: MSG00498 (CLM100697)...
  Classifying 5/75: MSG05703 (CLM100326)...
  Classifying 6/75: MSG06452 (CLM100495)...
  Classifying 7/75: MSG02448 (CLM100067)...
  Classifying 8/75: MSG03590 (CLM100016)...
  Classifying 9/75: MSG05364 (CLM100786)...
  Classifying 10/75: MSG07120 (CLM100776)...
  Classifying 11/75: MSG05088 (CLM100051)...
  Classifying 12/75: MSG07411 (CLM100363)...
  Classifying 13/75: MSG01327 (CLM100337)...
  Classifying 14/75: MSG04264 (CLM100477)...
  Classifying 15/75: MSG00294 (CLM100886)...
  Classifying 16/75: MSG01502 (CLM100438)...
  Classifying 17/75: MSG00885 (CLM100045)...
  Classifying 18/75: MSG00321 (CLM100354)...
  Classifying 19/75: MSG06535 (CLM100678)...
  C

In [8]:
## automatically scheduling follow-ups

exec(open("followup scheduler.py").read())

Follow-up decisions computed for 900 claims (reference date: 2026-09-16)

Action breakdown:
action
schedule_followup              586
no_followup_needed             236
recommend_closure               52
handled_by_total_loss_track     26
Name: count, dtype: int64

Follow-up business-day breakdown (scheduled only):
follow_up_business_days
1.0      30
2.0      50
3.0     282
5.0     208
10.0     16
Name: count, dtype: int64

Recommended for closure (no response 20+ business days): 52

Saved to followup_queue.csv


In [9]:
## categorizing adjuster vs ai work

exec(open("adjuster queue split.py").read())

Adjuster queue built from 75 tasks

Adjuster-facing bucket breakdown:
adjuster_bucket
needs_review    47
call_needed     18
ai_handled      10
Name: count, dtype: int64

Actually in the adjuster's queue: 65
Handled by AI assistant (documented, not in queue): 10

AI-handled breakdown by original contact_method:
contact_method
refer_total_loss_adjuster    8
refer_attorney_desk          2
Name: count, dtype: int64

Saved full queue (with documentation) to adjuster_queue.csv
